This notebook is designed to return a dataframe of each team's record, by season. This is intended to be used to help support offseason regression with respect to elo ratings. 

In [11]:
import pandas as pd
pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)
import numpy as np 
import matplotlib.pyplot as plt 

clean = pd.read_parquet("../data/games_clean.parquet")
print(clean.shape)
print(clean.columns)

(3829, 21)
Index(['game_id', 'season', 'week', 'kickoff', 'game_type', 'is_playoff',
       'home_team', 'away_team', 'location', 'is_neutral', 'home_score',
       'away_score', 'result', 'home_win', 'home_rest', 'away_rest',
       'home_qb_id', 'away_qb_id', 'spread_line', 'home_moneyline',
       'away_moneyline'],
      dtype='str')


In [32]:
reg_played = clean[(clean["game_type"] == "REG")& clean["home_score"].notna()]

home = pd.DataFrame({
    "season": reg_played["season"],
    "team": reg_played["home_team"],
    "pf": reg_played["home_score"],
    "pa": reg_played["away_score"],
})

away = pd.DataFrame({
    "season": reg_played["season"],
    "team": reg_played["away_team"],
    "pf": reg_played["away_score"],
    "pa": reg_played["home_score"],
})

team_games = pd.concat([home,away], ignore_index=True)
print(team_games.shape)
team_games["win"] = (team_games["pf"] > team_games["pa"]).astype(int)
team_games["loss"] = (team_games["pf"] < team_games["pa"]).astype(int)
team_games["tie"] = (team_games["pf"] == team_games["pa"]).astype(int)

records = (
    team_games.groupby(["season", "team"])
    .agg(games=("win", "count"), wins=("win", "sum"),
         losses=("loss", "sum"), ties=("tie", "sum"))
    .reset_index()
)

records["win_pct"] = (records["wins"] + 0.5*records["ties"]) / records["games"]
records.sort_values(["season", "win_pct"], ascending=[True, False], inplace=True)
records.reset_index(drop=True,inplace=True)
records.tail(128)

(7326, 4)


,season,team,games,wins,losses,ties,win_pct
320,2022,KC,17,14,3,0,0.823529
321,2022,PHI,17,14,3,0,0.823529
322,2022,BUF,16,13,3,0,0.812500
323,2022,MIN,17,13,4,0,0.764706
324,2022,SF,17,13,4,0,0.764706
325,2022,CIN,16,12,4,0,0.750000
326,2022,DAL,17,12,5,0,0.705882
327,2022,BAL,17,10,7,0,0.588235
328,2022,LAC,17,10,7,0,0.588235
329,2022,NYG,17,9,7,1,0.558824
